RAG pipeline.  Source text: PDFs from UK Financial Conduct Authority (FCA) online [Handbook](https://handbook.fca.org.uk/handbook), specifically the Conduct of Business Sourcebook (COBS) section, chapters 1-10A, last updated on 5 August 2026. 

First install requirements.

In [0]:
%pip install \
    langchain==1.3.16 \
    langchain-chroma==1.1.0 \
    langchain-groq==1.1.3 \
    langchain-huggingface==1.2.2 \
    huggingface_hub==1.28.0 \
    sentence-transformers==6.0.0
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader # langchain-community sunsetting, fix this later
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

DATA_PATH = "fca_cobs_pdfs"
SAVE_PATH = "chroma_index" 
MODEL_NAME = "all-MiniLM-L6-v2"

# load PDFs
loader = PyPDFDirectoryLoader(DATA_PATH)
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDFs.")

In [0]:
def clean_fca_text(text):
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = line.strip()
        # skip lines that are just R, G, N or COBS (case insensitive)
        if line.upper() in ['R', 'G', 'N', 'COBS', 'CHAPTER']:
            continue
        # skip lines that are just dots or whitespace
        if not line or re.match(r'^[\.\s\-_]+$', line):
            continue
        # skip lines that look like the handbook header/footer
        if 'www.handbook.fca.org.uk' in line or re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s\d{4}', line):
            continue
        cleaned_lines.append(line)
    # join back with single newlines
    return '\n'.join(cleaned_lines)

In [0]:
for doc in docs:
    doc.page_content = clean_fca_text(doc.page_content)

# chunk documents
# 1000 character chunk with 200 overlap to keep legal context intact
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")

import random
print(random.choice(chunks).page_content)

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# initialise embedding model
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)

# create Chroma vector store and save locally
vector_store = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings, 
    persist_directory=SAVE_PATH
)

print(f"Vector store successfully saved.")


In [0]:
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")


In [0]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# initialise components, reload vector_store from saved, embeddings as above
# vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
vector_store = Chroma(persist_directory=SAVE_PATH, embedding_function=embeddings)
llm = ChatGroq(model_name="groq/compound-mini", temperature=0)

# prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional UK Financial Compliance Assistant. 
Answer the user's question using ONLY the provided context from the FCA COBS handbook (Chapters 1-10A).
    
CRITICAL INSTRUCTIONS:
1. Do NOT start your answer with phrases like "Based on the context," "Based on the provided text," or "According to the documents."
2. Start your answer **directly** with the factual information.
3. If the answer is not in the context, simply state: "I cannot find this information in the provided COBS chapters."
4. Do not hallucinate. Be precise and professional.

Context: {context}
Question: {question}

Helpful answer:"""),
    ("human", "{question}")
])

# build LCEL Chain
# format prompt -> pass to LLM -> parse output as string
retriever = vector_store.as_retriever(search_kwargs={"k": 6})

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

In [0]:
query = "What are the requirements for client categorisation?"
response = rag_chain.invoke(query)

print(response)

In [0]:
query = "What are the rules on inducements?"
response = rag_chain.invoke(query)

print(response)

In [0]:
query = "Which potential clients might need enhanced KYC?"
response = rag_chain.invoke(query)

print(response)

In [0]:
query = "What is the first line of COBS 1?"
response = rag_chain.invoke(query)

print(response)